# 010 Final Recap: How a Skill Grows from 0 to Usable

这是第十课：完整复盘这份天气 Skill 是怎么从 0 长到可用的。

学习目标：

1. 复盘这条 Skill 教学线每一课到底解决了什么问题
2. 理解一份 Skill 为什么不应该一开始就做成“大而全”
3. 提炼出一套以后还可以复用的 Skill 演进顺序
4. 把这份天气 Skill 从结构、流程、验证三个维度完整看一遍

这节课继续使用：

- `.agents/skills/weather-query-assistant/`


## 先明确这节课的作用

前面 9 课已经把这份天气 Skill 从零一路做到了可用。

第十课不再加新能力。

它只做一件事：

- 把这条成长路径彻底梳清楚

这一步很重要，因为如果你不做复盘，很容易学完一堆局部动作，却不知道它们为什么要按这个顺序出现。


## 先看最终真实目录

先看结果，再回头看过程。


In [1]:
from pathlib import Path


skill_root = Path('.agents/skills/weather-query-assistant')
print('exists =', skill_root.exists())
print('path =', skill_root.resolve())


exists = True
path = /home/dev/bxc/fastapi-study/.agents/skills/weather-query-assistant


In [2]:
def print_tree(root: Path, prefix: str = '') -> None:
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for index, entry in enumerate(entries):
        connector = '└── ' if index == len(entries) - 1 else '├── '
        print(prefix + connector + entry.name)
        if entry.is_dir():
            next_prefix = prefix + ('    ' if index == len(entries) - 1 else '│   ')
            print_tree(entry, next_prefix)


print(skill_root)
print_tree(skill_root)


.agents/skills/weather-query-assistant
├── agents
│   └── openai.yaml
├── references
│   └── weather_sources.md
├── scripts
│   ├── __pycache__
│   │   ├── build_open_meteo_query.cpython-310.pyc
│   │   ├── build_open_meteo_query.cpython-313.pyc
│   │   ├── build_wttr_query.cpython-310.pyc
│   │   ├── build_wttr_query.cpython-313.pyc
│   │   ├── normalize_location.cpython-310.pyc
│   │   └── normalize_location.cpython-313.pyc
│   ├── build_open_meteo_query.py
│   ├── build_wttr_query.py
│   ├── normalize_location.py
│   └── validate_weather_skill.py
└── SKILL.md


## 先看最终版 `SKILL.md`

你现在再读这份 `SKILL.md`，应该已经不会只把它当成一份说明文字了。

你应该能看出：

- 它定义了 workflow
- 它引用了 references
- 它约定了 scripts
- 它已经是一个真正可执行工作流的入口


In [3]:
print((skill_root / 'SKILL.md').read_text(encoding='utf-8'))


---
name: weather
description: Get current weather and forecasts (no API key required).
homepage: https://wttr.in/:help
metadata: {"nanobot":{"emoji":"🌤️","requires":{"bins":["curl"]}}}
---

# Weather

Use this skill when the user asks for current weather or a short forecast for a specific location.

## Workflow

1. Check whether the user gave a clear location.
2. If the location is missing or ambiguous, ask a short clarification question.
3. Normalize the location when the user input is noisy or inconsistently formatted.
4. Build a stable weather query string before calling the external service.
5. Use `wttr.in` as the primary source.
6. Use Open-Meteo as a fallback when JSON output or more programmatic structure is needed.
7. Answer concisely with practical details:
   - location
   - current condition or forecast summary
   - temperature
   - humidity or wind when relevant
   - rain risk when relevant
8. Do not guess when weather data is unavailable.

## References

- Read `referenc

## 这 10 课到底各自解决了什么

这一步必须讲清楚。

否则你很容易只记住“加了哪些文件”，却记不住“为什么是这个顺序”。


In [4]:
lesson_map = [
    (1, '读真实 Skill', '先建立真实目录和文件职责的直觉'),
    (2, '新建最小 Skill', '学会先做最小可用结构'),
    (3, '引入 references', '把资料细节从 SKILL.md 下沉出去'),
    (4, '引入 scripts', '把第一个重复且确定性强的动作脚本化'),
    (5, '接成 mini workflow', '让多个脚本开始协同工作'),
    (6, '跑完整小会话', '从用户问题一路走到可执行命令'),
    (7, '处理错误和 fallback', '让 Skill 开始具备异常分支意识'),
    (8, '实现真实 fallback', '让 Open-Meteo 路径从概念变成实现'),
    (9, '建立最小验证路径', '让 Skill 具备快速回归检查能力'),
    (10, '完整复盘', '提炼以后还可复用的 Skill 演进方法'),
]

from pprint import pprint
pprint(lesson_map)


[(1, '读真实 Skill', '先建立真实目录和文件职责的直觉'),
 (2, '新建最小 Skill', '学会先做最小可用结构'),
 (3, '引入 references', '把资料细节从 SKILL.md 下沉出去'),
 (4, '引入 scripts', '把第一个重复且确定性强的动作脚本化'),
 (5, '接成 mini workflow', '让多个脚本开始协同工作'),
 (6, '跑完整小会话', '从用户问题一路走到可执行命令'),
 (7, '处理错误和 fallback', '让 Skill 开始具备异常分支意识'),
 (8, '实现真实 fallback', '让 Open-Meteo 路径从概念变成实现'),
 (9, '建立最小验证路径', '让 Skill 具备快速回归检查能力'),
 (10, '完整复盘', '提炼以后还可复用的 Skill 演进方法')]


## 为什么这个顺序比“一次性做全”更好

这是这条教学线最核心的工程观。

如果一开始你就把下面这些东西一次性全做出来：

- references
- scripts
- fallback
- validation
- 各种目录

短期看起来会很完整，长期反而更乱。

因为你还没搞清楚：

- 哪些东西真的需要存在
- 哪些边界已经稳定
- 哪些目录只是“预支复杂度”

所以这条路线的价值，不只是教你做天气 Skill，而是教你控制复杂度增长的顺序。


In [5]:
why_incremental_growth_is_better = [
    'you only add structure when there is a real need',
    'boundaries get clearer before you abstract them',
    'maintenance cost stays visible',
    'you avoid empty or premature directories',
]

pprint(why_incremental_growth_is_better)


['you only add structure when there is a real need',
 'boundaries get clearer before you abstract them',
 'maintenance cost stays visible',
 'you avoid empty or premature directories']


## 现在把这份 Skill 分成三个层次来看

如果从最终形态回看，这份天气 Skill 大致可以分成三层。


In [6]:
skill_layers = {
    'workflow_layer': ['SKILL.md'],
    'reference_layer': ['references/weather_sources.md'],
    'execution_layer': [
        'scripts/normalize_location.py',
        'scripts/build_wttr_query.py',
        'scripts/build_open_meteo_query.py',
        'scripts/validate_weather_skill.py',
    ],
}

pprint(skill_layers)


{'execution_layer': ['scripts/normalize_location.py',
                     'scripts/build_wttr_query.py',
                     'scripts/build_open_meteo_query.py',
                     'scripts/validate_weather_skill.py'],
 'reference_layer': ['references/weather_sources.md'],
 'workflow_layer': ['SKILL.md']}


## 这三个层次分别负责什么

把职责分清楚，是你以后继续做 Skill 的关键。

- `SKILL.md`
  负责回答：什么时候用、按什么顺序做

- `references/`
  负责回答：需要时去哪里查具体资料

- `scripts/`
  负责回答：哪些重复而确定的动作可以直接执行

如果这三层混在一起，Skill 会越来越难维护。


In [ ]:
layer_roles = {
    'SKILL.md': 'workflow and trigger entry',
    'references/': 'lookup material and concrete details',
    'scripts/': 'repeatable deterministic actions',
}

pprint(layer_roles)


{'SKILL.md': 'workflow and trigger entry',
 'references/': 'lookup material and concrete details',
 'scripts/': 'repeatable deterministic actions'}


{'SKILL.md': 'workflow and trigger entry',
 'references/': 'lookup material and concrete details',
 'scripts/': 'repeatable deterministic actions'}


## 这份天气 Skill 的最终能力边界是什么

到现在为止，这份 Skill 已经能做到：

1. 识别天气查询工作流
2. 处理地点澄清分支
3. 标准化地点输入
4. 构建 `wttr.in` 主路径查询
5. 构建 Open-Meteo fallback 查询
6. 做最小验证

但它依然没有做这些事情：

- 完整地理编码系统
- 复杂天气解释
- 长篇旅行建议
- 完整测试框架

这不是缺点，而是边界控制。


In [ ]:
final_scope = {
    'does': [
        'workflow routing',
        'clarification path',
        'location normalization',
        'primary query building',
        'fallback query building',
        'minimal validation',
    ],
    'does_not': [
        'full geocoding system',
        'deep weather analytics',
        'travel recommendation engine',
        'full test framework',
    ],
}

pprint(final_scope)


{'does': ['workflow routing',
          'clarification path',
          'location normalization',
          'primary query building',
          'fallback query building',
          'minimal validation'],
 'does_not': ['full geocoding system',
              'deep weather analytics',
              'travel recommendation engine',
              'full test framework']}


## 如果以后你要做别的 Skill，可以直接复用什么方法

这才是第十课最值得带走的东西。

不是“天气 Skill 长什么样”，而是“别的 Skill 也可以按这个顺序长”。


In [ ]:
reusable_skill_growth_pattern = [
    'start with the smallest useful SKILL.md',
    'add metadata when presentation matters',
    'move bulky details into references when needed',
    'script only stable repeated actions',
    'connect scripts into a mini workflow',
    'teach or design fallback paths explicitly',
    'add a minimum validation path before calling it stable',
]

pprint(reusable_skill_growth_pattern)


## 什么情况下你现在就该停，不要继续扩展

很多人学到这里会有冲动：

- 再加更多脚本
- 再接更多 API
- 再做更复杂的推理

但正式开发里，知道什么时候停同样重要。

如果一份 Skill 已经满足：

- workflow 清楚
- 路径清楚
- fallback 清楚
- 验证清楚

那它就已经到了一个很好的“可用版本”。


In [ ]:
stop_signals = [
    'workflow is clear',
    'core paths are implemented',
    'fallback path exists',
    'validation exists',
    'new additions would mostly add complexity instead of clarity',
]

pprint(stop_signals)


## 这条 Skill 教学线到这里为什么可以收尾

因为现在这份天气 Skill 已经具备了一条完整且自洽的成长链：

- 从最小结构开始
- 再引入资料
- 再引入脚本
- 再引入路径分支
- 再引入验证

如果继续往后加内容，收益就开始递减了。

所以第十课的意义不是“结束内容”，而是“把方法沉淀下来”。


## 最后的工程化总结

如果把这 10 课压成几句最值得带走的话，就是：

1. Skill 不是目录越多越好，而是边界越清楚越好
2. 不要一开始就预支复杂度
3. 先让 `SKILL.md` 可用，再逐步补 `references/` 和 `scripts/`
4. fallback 和验证不是可有可无，它们决定一份 Skill 是否真正可用
5. 一份好的 Skill，本质上是一条清晰、可维护、可验证的小工作流


In [ ]:
final_takeaways = [
    'clarity over directory count',
    'incremental growth over premature completeness',
    'workflow first, then references and scripts',
    'fallback and validation make a skill truly usable',
    'a good skill is a maintainable mini workflow',
]

pprint(final_takeaways)


## 当前阶段结论

你现在需要记住：

1. 这条天气 Skill 教学线已经完整闭环
2. 你真正学到的不是一个天气 Skill，而是一套 Skill 演进方法
3. 以后做别的 Skill，也应该优先复用这套“按需增长”的顺序
4. 一份 Skill 到了 workflow、fallback、validation 都清楚的阶段，就已经是一个很好的可用版本
5. 下一步不必马上继续加复杂度，更值得把这套方法迁移到另一个新 Skill 上

下一步建议：

- 把这套方法迁移到另一个新 Skill
- 或者回头优化 `AGENTS.md`，把这条 Skill 教学线沉淀成长期记忆
